In [ ]:
!git clone https://github.com/mahmoodlab/MI-Zero.git
!git clone https://github.com/mahmoodlab/CLAM.git


In [ ]:
%cd MI-Zero
!pip install assets/timm_ctp.tar --no-deps

In [ ]:
!pwd

In [ ]:
import pandas as pd
import numpy as np
import os
import torch
import torch.nn as nn
import h5py
from torch.utils.data import DataLoader, Dataset
from torchvision import transforms
import torch.nn.functional as F
import sys
import math

from src.models.ctran import ctranspath

In [ ]:
model = ctranspath(img_size = 224)

In [ ]:
model

In [ ]:
def load_ctranspath_clip(ckpt_path, img_size = 224, return_trsforms = True):
    def clean_state_dict_clip(state_dict):
        new_state_dict = {}
        for k, v in state_dict.items():
            if 'attn_mask' in k:
                continue
            if 'visual.trunk.' in k:
                new_state_dict[k.replace('module.visual.trunk.', '')] = v
        return new_state_dict
    
    model = ctranspath(img_size = img_size)
    model.head = nn.Identity()
    state_dict = torch.load(ckpt_path, map_location="cpu")['state_dict']
    state_dict = clean_state_dict_clip(state_dict)
    missing_keys, unexpected_keys = model.load_state_dict(state_dict, strict=False)
    print('missing keys: ', missing_keys)
    print('unexpected keys: ', unexpected_keys)
    
    if return_trsforms:
        trsforms = get_transforms_ctranspath(img_size = img_size)
        return model, trsforms
    return model
def get_transforms_ctranspath(img_size=224, 
                              mean = (0.485, 0.456, 0.406), 
                              std = (0.229, 0.224, 0.225)):
    trnsfrms = transforms.Compose(
                    [
                     transforms.Resize(img_size),
                     transforms.ToTensor(),
                     transforms.Normalize(mean = mean, std = std)
                    ]
                )
    return trnsfrms
ckpt_path = "/kaggle/input/mizerow/ctranspath_448_bioclinicalbert/ctranspath_448_bioclinicalbert/checkpoints/epoch_50.pt"
model, trsforms = load_ctranspath_clip(ckpt_path, img_size = 224)

In [ ]:
import openslide
wsi_path = "/kaggle/input/prostate-cancer-grade-assessment/train_images/0005f7aaab2800f6170c399693a96917.tiff"
wsi = openslide.open_slide(wsi_path)

In [ ]:
!mkdir ../debug 
!mkdir ../PATCH 
!cp /kaggle/input/prostate-cancer-grade-assessment/train_images/0005f7aaab2800f6170c399693a96917.tiff ../debug/

In [ ]:
!python ../CLAM/create_patches_fp.py --source ../debug --save_dir ../PATCH --patch_size 256 --seg --patch --stitch 

In [ ]:
!ls ../PATCH/

In [ ]:
import cv2
import matplotlib.pyplot as plt

mask = cv2.imread("../PATCH/stitches/0005f7aaab2800f6170c399693a96917.jpg")
plt.imshow(mask)
plt.show()

In [ ]:
img = cv2.imread("../PATCH/masks/0005f7aaab2800f6170c399693a96917.jpg")
plt.imshow(img)
plt.show()

In [ ]:
def read_assets_from_h5(h5_path):
    assets = {}
    attrs = {}
    with h5py.File(h5_path, 'r') as f:
        for key in f.keys():
            assets[key] = f[key][:]
            if f[key].attrs is not None:
                attrs[key] = dict(f[key].attrs)
    return assets, attrs

assets, _ = read_assets_from_h5("../PATCH/patches/0005f7aaab2800f6170c399693a96917.h5")

In [ ]:
class Whole_Slide_Bag_FP(Dataset):
    def __init__(self,
        coords,
        wsi,
        patch_level,
        patch_size,
        custom_transforms=None,
        target_patch_size=-1):
        """
        Args:
            coords (string): coordinates to extract patches from w.r.t. level 0.
            custom_transforms (callable, optional): Optional transform to be applied on a sample
            target_patch_size (int): Custom defined image size before embedding
        """
        self.coords = coords
        self.patch_level = patch_level
        self.patch_size = patch_size
        self.wsi = wsi
        self.roi_transforms = custom_transforms
        self.length = len(coords)
        if target_patch_size > 0:
            self.target_patch_size = (target_patch_size, ) * 2
        else:
            self.target_patch_size = None

    def __len__(self):
        return self.length
    
    def __getitem__(self, idx):
        coord = self.coords[idx]
        img = self.wsi.read_region(coord, self.patch_level, (self.patch_size, self.patch_size)).convert('RGB')
        if self.target_patch_size is not None:
            img = img.resize(self.target_patch_size)
        img = self.roi_transforms(img)
        return {'img': img, 'coords': coord}

In [ ]:
dataset = Whole_Slide_Bag_FP(assets["coords"],
                                 wsi,
                                 0,
                                 patch_size=256,
                                 custom_transforms = trsforms,
                                 target_patch_size = 256)
dataloader = DataLoader(dataset, shuffle=False, batch_size=32)
features_ = []
device = "cpu"
coords_ = []
with torch.inference_mode():
    for batch in dataloader:
        imgs = batch['img'].to(device)
        coords = np.array(batch['coords'])
        features = model(imgs)
        features_.append(features)
        coords_.append(coords)
        



In [ ]:
coords_ = np.concatenate(coords_)
features_ = torch.cat(features_)

In [ ]:
def tokenize(tokenizer, texts=["a histopathological image of adipocytes"]):
    tokens = tokenizer.batch_encode_plus(texts, 
                                         max_length = 64,
                                         add_special_tokens=True, # Add '[CLS]' and '[SEP]'
                                         return_token_type_ids=False,
                                         truncation = True,
                                         padding = 'max_length',
                                         return_attention_mask=True)
    return tokens['input_ids'], tokens['attention_mask']
from transformers import AutoTokenizer
model_name = 'emilyalsentzer/Bio_ClinicalBERT'
tokenizer = AutoTokenizer.from_pretrained(model_name, fast=True)

In [ ]:
from src.models.model import CLIP
import json
IMAGENET_COLOR_MEAN = (0.485, 0.456, 0.406)
IMAGENET_COLOR_STD = (0.229, 0.224, 0.225)

def clean_state_dict_ctranspath(state_dict):
    new_state_dict = {}
    for k, v in state_dict.items():
        if 'attn_mask' in k:
            continue
        new_state_dict[k.replace('module.', '')] = v
    return new_state_dict

def create_model(
        device: torch.device = torch.device('cpu'),
        override_image_size = None,
        model_checkpoint = None,
):
    with open("./src/model_configs/ctranspath_448_bioclinicalbert.json") as f:
        model_cfg = json.load(f)
    
    if override_image_size:
        model_cfg['vision_cfg']['image_size'] = override_image_size
        logging.info(f'Created model {model_name} with image size of {override_image_size} instead')
        
    model = CLIP(**model_cfg)
    
    model.to(device=device)
    model.visual.image_mean = IMAGENET_COLOR_MEAN
    model.visual.image_std = IMAGENET_COLOR_STD
    state_dict = torch.load(model_checkpoint, map_location='cpu')['state_dict']
    state_dict = clean_state_dict_ctranspath(state_dict)
    missing_keys, _ = model.load_state_dict(state_dict, strict=False)
    print(missing_keys)
    return model

In [ ]:

clip_model = create_model(model_checkpoint = ckpt_path)
image_features = clip_model.visual.head(features_) # Project to desired dimensions num_patches x 512
image_features = F.normalize(image_features, dim=-1) 



In [ ]:
texts, attention_mask = tokenize(tokenizer,texts=["a histopathological image of prostate cancer"]) # Tokenize with custom tokenizer
texts = torch.from_numpy(np.array(texts))#.to(device)
attention_mask = torch.from_numpy(np.array(attention_mask)).to(device)
class_embeddings = clip_model.encode_text(texts, attention_mask=attention_mask)

class_embedding = F.normalize(class_embeddings, dim=-1)

In [ ]:
texts, attention_mask = tokenize(tokenizer,texts=["a histopathological image of adipose tissue"]) # Tokenize with custom tokenizer
texts = torch.from_numpy(np.array(texts))#.to(device)
attention_mask = torch.from_numpy(np.array(attention_mask)).to(device)
class_embeddings2 = clip_model.encode_text(texts, attention_mask=attention_mask)

class_embedding2 = F.normalize(class_embeddings2, dim=-1)

In [ ]:
texts, attention_mask = tokenize(tokenizer,texts=["adipose"]) # Tokenize with custom tokenizer
texts = torch.from_numpy(np.array(texts))#.to(device)
attention_mask = torch.from_numpy(np.array(attention_mask)).to(device)
class_embedding3 = clip_model.encode_text(texts, attention_mask=attention_mask)

class_embedding3 = F.normalize(class_embedding3, dim=-1)

In [ ]:
texts, attention_mask = tokenize(tokenizer,texts=["stromal"]) # Tokenize with custom tokenizer
texts = torch.from_numpy(np.array(texts))#.to(device)
attention_mask = torch.from_numpy(np.array(attention_mask)).to(device)
class_embedding_stromal = clip_model.encode_text(texts, attention_mask=attention_mask)

class_embedding_stromal = F.normalize(class_embedding_stromal, dim=-1)

In [ ]:
image_features.shape

In [ ]:
class_embedding.shape

In [ ]:

import torch.nn as nn
cos = nn.CosineSimilarity(dim=1, eps=1e-6)
output = cos(image_features, class_embedding)

In [ ]:
def get_patch_img(wsi,coords,cos_sim,index):
    coord = coords[index]
    sim = cos_sim[index]
    img = wsi.read_region(coord, 0, (256,256)).convert('RGB')
    plt.title(f"cos sim: {sim}")
    plt.imshow(img)
    plt.show()
    plt.close()

In [ ]:
for idx in torch.argsort(output).numpy()[::-1][:10]:
    get_patch_img(wsi,assets["coords"],output.detach().numpy(),idx)

In [ ]:
cos = nn.CosineSimilarity(dim=1, eps=1e-6)
output2 = cos(image_features, class_embedding2)

for idx in torch.argsort(output2).numpy()[::-1][:10]:
    get_patch_img(wsi,assets["coords"],output2.detach().numpy(),idx)

In [ ]:
cos = nn.CosineSimilarity(dim=1, eps=1e-6)
output3 = cos(image_features, class_embedding3)

for idx in torch.argsort(output3).numpy()[::-1][:10]:
    get_patch_img(wsi,assets["coords"],output3.detach().numpy(),idx)

In [ ]:
cos = nn.CosineSimilarity(dim=1, eps=1e-6)
output_stromal = cos(image_features, class_embedding_stromal)

for idx in torch.argsort(output_stromal).numpy()[::-1][:10]:
    get_patch_img(wsi,assets["coords"],output_stromal.detach().numpy(),idx)